In [1]:
from typing import List
from mandala.all import *
set_logging_level('warning')

# create a storage for results
storage = Storage(in_memory=True) # can also be persistent (on disk)

@op(storage) # memoization decorator
def inc(x) -> int:
    return x + 1 

@op(storage) 
def mean(x:List[int]) -> float: 
    # you can operate on / return collections of memoized results
    return sum(x) / len(x) 

with run(storage): # calls inside `run` block are memoized
    nums = [inc(i) for i in range(5)]
    result = mean(nums) # memoization composes through lists without copying data
    print(f'Mean of 5 nums: {result}')

# add logic/parameters directly on top of memoized code without re-doing past work
with run(storage, lazy=True):
    nums = [inc(i) for i in range(10)]
    result = mean(nums) 

# walk over chains of calls without loading intermediate data 
# to traverse storage and collect results flexibly
with run(storage, lazy=True):
    nums = [inc(i) for i in range(10)]
    result = mean(nums) 
    print(f'Reference to mean of 10 nums: {result}')
    storage.attach(result) # load the value in-place 
    print(f'Loaded mean of 10 nums: {result}')

# pattern-match to memoized compositions of calls
with query(storage) as q: 
    # this may not make sense unless you read the tutorials
    i = Query()
    inc_i = inc(i).named('inc_i')
    nums = MakeList(containing=inc_i, at_index=0).named('nums')
    result = mean(nums).named('result')
    df = q.get_table(inc_i, nums, result)
df

Mean of 5 nums: AtomRef(3.0, type=AtomType(float))


Reference to mean of 10 nums: AtomRef(in_memory=False, type=AtomType(float))


Loaded mean of 10 nums: AtomRef(5.5, type=AtomType(float))


,inc_i,nums,result
0,1,"[1, 2, 3, 4, 5]",3.0
1,1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",5.5
